# Hyperliquid consolidated account analytics

Combines balances, account history, fills, funding, positions, and PnL across the configured Hyperliquid wallets. Combined account-value history is shown only over the dates available for every wallet; charts show the oldest wallet's account history before shared coverage begins, then the combined account value.

Daily rollups and calendar-period cutoffs use `America/New_York`. Previously saved UTC-aggregated daily totals are mapped to Eastern date labels when loaded; their original intraday split cannot be reconstructed.


## Wallets and implementation

Edit `WALLET_ADDRESSES` here. Both addresses found in the single-wallet notebook are included: the active address and the previously commented address.


In [1]:
import json
from pathlib import Path

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]

RISK_BUDGET_PCT = 5.0
ATR_PERIOD = 14
ATR_STOP_MULTIPLE = 1.25
MARGIN_RESERVE_PCT = 20.0
namespace={'__name__':'hyperliquid_multi_account_analytics'}
implementation_paths=[
    Path('hyperliquid_account_analytics_implementation.ipynb'),
    Path('hyperliquid_multi_account_analytics_implementation.ipynb'),
]
for implementation_path in implementation_paths:
    implementation=json.loads(implementation_path.read_text(encoding='utf-8'))
    for cell in implementation['cells']:
        if cell['cell_type']=='code':
            source=''.join(cell['source'])
            if source.strip():
                exec(compile(source,f'{implementation_path} cell','exec'),namespace)
MultiWalletAnalytics=namespace['MultiWalletAnalytics']
analytics=MultiWalletAnalytics(WALLET_ADDRESSES,data_dir='data')
analytics.position_risk_budget_pct=RISK_BUDGET_PCT
analytics.position_atr_period=ATR_PERIOD
analytics.position_atr_stop_multiple=ATR_STOP_MULTIPLE
analytics.margin_reserve_pct=MARGIN_RESERVE_PCT


## Fetch and consolidate wallet data

Fetch each wallet once and prepare combined account history and fills.


In [2]:
%%capture
_ = analytics.fetch_data()


## Account / PnL

Combined account and risk summary. Daily PnL and calendar-period cutoffs use US Eastern time.


In [3]:
analytics.show_account_pnl()


=== ACCOUNT / PNL AND RISK ===


Metric,Value
Account value,"$2,918.78"
Portfolio value,"$3,613.49 (combined live wallet value)"
Account 1D (ET),2.39%
Realized today (ET),$1.27
Realized today (% of account value),0.04%
Unrealized PnL,$-106.87
Unrealized PnL (% of account value),-3.66%
Unrealized 1D change (ET),n/a
Average realized PnL / day,$-0.96
Average unrealized PnL change / day,n/a


## Account charts

Account value and PnL charts. Before all wallets have saved history, account value and weekly returns use the oldest wallet alone; from shared coverage onward they use the combined accounts.


In [4]:
analytics.show_account_charts()


Weekly return chart omits 2 return(s) below -100%;                     other data and stats are unchanged.


## Account vs BTC

Daily account-vs-BTC comparison uses the oldest wallet before shared history begins, then the combined accounts. The lifetime realized-PnL comparison pools the available wallet history.


In [5]:
analytics.show_account_vs_btc()


Lifetime account series is anchored to current account value; it includes fee-net realized trades and perp funding, but excludes historical cash flows and unrealized PnL.


## Traded volume vs realized PnL chart

Daily fill volume and realized PnL pooled across wallets.


In [6]:
analytics.show_volume_chart()


## Open positions

Show each wallet position separately, followed by fixed-fractional sizing grouped by asset and side and an aggregate summary. The summary includes cumulative open-position funding as a percentage of combined account equity and total position value. Reserve-adjusted available margin is combined account equity less the configured reserve; the summary shows that capacity, margin used, remaining available margin in dollars and as a percentage of that capacity, and position value as a percentage of equity. Set `MARGIN_RESERVE_PCT` (20% by default) in the setup cell to change the reserve. The position table includes funding and fees since open as dollar amounts, plus fees since open as percentages of wallet equity and position value and position value as a percentage of wallet account value. Fee ratios are available for complete USDC-denominated fee history. Set `RISK_BUDGET_PCT` (5% per asset), `ATR_PERIOD` (14 completed daily candles), and `ATR_STOP_MULTIPLE` (1.25) in the setup cell. Stops are placed 1.25 ATR from the current mark, and take-profit is set the same distance in the favorable direction (1:1 risk/reward). Same-side positions across wallets are combined; long and short rows remain separate. If both sides are open for one asset, they share that asset's risk budget equally.


In [7]:
analytics.show_open_positions()


Wallet,Coin,Dex,Side,Liq %,ROE,uPnL,Funding (since open),Funding (1h),Funding APR,Expected Price PnL (5d),Expected Funding PnL (5d),Expected PnL incl. funding (5d),Entry,Mark,Pos,Margin,Margin Type,Margin %,Fees,Fees / Equity (%),Fees / Position (%),Position / Account (%),Liq Px,Lev.,Sz
c248,IMX,,SHORT,56.55%,8.01%,$15.89,$0.54,0.001250%,10.95%,$12.32,$1.46,$13.78,0.12332,0.12135,$976,$195,Cross,29.04%,-$0.14,0.015%,0.015%,100.28%,0.18998,5.00x,"-8,040.50"
07f2,TRUMP,,SHORT,172.21%,16.76%,$32.08,$0.00,0.000989%,8.66%,$11.47,$0.29,$11.76,1.75136,1.6769,$734,$183,Cross,12.54%,-$0.20,0.008%,0.027%,27.79%,4.5646,4.00x,-437.30
07f2,TRX,,SHORT,180.40%,-4.31%,-$3.01,$0.03,0.001250%,10.95%,-$1.44,-$1.12,-$2.55,0.313,0.31435,$700,$70,Cross,4.78%,-$0.10,0.004%,0.014%,26.50%,0.88143,10.00x,"-2,226.00"
07f2,PENGU,,SHORT,190.85%,1.47%,$1.86,$0.24,0.001250%,10.95%,$2.11,$0.51,$2.62,0.007292,0.007271,$631,$126,Cross,8.63%,-$0.09,0.003%,0.014%,23.91%,0.021148,5.00x,"-86,832.00"
07f2,xyz:GOLD,xyz,LONG,n/a,-0.13%,-$0.11,-$0.05,0.002110%,18.48%,-$0.99,-$0.66,-$1.65,"3,909.8","3,909",$631,$90,Cross,6.16%,-$0.18,0.007%,0.029%,23.88%,0,7.00x,0.16
07f2,BTC,,LONG,n/a,-0.44%,-$0.87,-$0.03,0.001250%,10.95%,-$1.63,-$0.79,-$2.42,"78,425","78,317",$598,$199,Cross,13.64%,-$0.09,0.003%,0.014%,22.66%,0,3.00x,0.01
07f2,xyz:GOOGL,xyz,LONG,n/a,-51.66%,-$100.24,-$2.33,0.004761%,41.71%,$0.12,-$0.65,-$0.53,343.02,283.95,$482,$161,Cross,10.98%,-$0.05,0.002%,0.010%,18.25%,0,3.00x,1.70
07f2,MELANIA,,SHORT,261.49%,1.17%,$1.70,$0.03,0.001250%,10.95%,-$0.44,$0.23,-$0.21,0.07686,0.07656,$434,$145,Cross,9.90%,-$0.06,0.002%,0.014%,16.45%,0.27676,3.00x,"-5,674.80"
07f2,io:PRL,io,SHORT,11.18%,-11.14%,-$15.21,$5.47,0.037267%,326.45%,$19.11,$11.86,$30.97,1.50057,1.5563,$425,$126,Isolated,8.63%,-$0.04,0.001%,0.009%,16.09%,1.7303,3.37x,-273.00
07f2,xyz:JIO,xyz,LONG,30.08%,-0.39%,-$0.43,$1.09,-0.299266%,-2621.57%,$25.94,$12.18,$38.12,14.54696,14.53,$333,$112,Isolated,7.64%,-$0.03,0.001%,0.009%,12.60%,10.16,2.98x,22.90


Funding and fees are cumulative since the position was opened. Fee ratios are shown only for complete USDC-denominated fees; otherwise they are n/a.
=== ACCOUNT NET EXPOSURE AND FUNDING ===


Net Direction,Long Positions,Short Positions,Long Notional ($),Short Notional ($),Gross Notional ($),Net Notional ($),Net Exposure / Gross (%),Net Funding Rate / 1h (%) (covered),"Net Funding APR (%) (simple, covered)",Estimated Net Funding PnL / hour ($),Estimated Net Funding PnL / day ($),Funding-Covered Notional ($),Funding Coverage (%)
NET SHORT,7,17,"$2,379.54","$6,003.89","$8,383.43","$-3,624.35",-43.23%,0.014013%,122.75%,$1.1747,$28.19,"$8,383.43",100.0%


Net direction compares long and short USD notionals; it is not delta- or beta-adjusted. Net funding rate is estimated net funding cash flow divided by funding-covered gross notional. Positive cash flow means estimated receipts. APR assumes current hourly rates persist; it is not a forecast.
=== FUNDING BY ASSET AND SIDE ===


Coin,Side,Positions,Size,Position Value ($),Current Funding / 1h (%),Funding APR (%),Estimated Funding PnL / hour ($),Estimated Funding PnL / day ($),Funding Coverage (%)
ADA,SHORT,1,"1,086",$208.87,0.001250%,10.95%,$0.0026,$0.06,100.0%
AERO,LONG,1,119,$74.67,0.001250%,10.95%,$-0.0009,$-0.02,100.0%
AVAX,SHORT,1,30,$259.31,0.001250%,10.95%,$0.0032,$0.08,100.0%
BTC,LONG,1,0.00764,$598.30,0.001250%,10.95%,$-0.0075,$-0.18,100.0%
FOGO,SHORT,1,7.942e+04,$204.74,0.001250%,10.95%,$0.0026,$0.06,100.0%
GRASS,SHORT,1,452.3,$271.80,0.001250%,10.95%,$0.0034,$0.08,100.0%
IMX,SHORT,1,"8,040",$975.71,0.001250%,10.95%,$0.0122,$0.29,100.0%
INIT,SHORT,1,"1,105",$193.42,0.001250%,10.95%,$0.0024,$0.06,100.0%
LIT,SHORT,1,9,$28.92,0.000250%,2.19%,$0.0001,$0.00,100.0%
MELANIA,SHORT,1,"5,675",$434.46,0.001250%,10.95%,$0.0054,$0.13,100.0%


Funding rates are current hourly rates, weighted by position notional across wallets. Positive rates are paid by longs and received by shorts; estimated cash flows assume the current rate stays constant. APR is simple annualization, not a forecast.
Fixed-fractional sizing: risk budget 5.00% of combined account value per asset; target quantity uses 1.25 x daily ATR(14) plus the directionally adverse 95th-percentile five-day funding reserve. Opposing sides, when both open, split the asset budget equally.
Limited daily history; ATR uses all completed candles available (shorter than 14 days) for: xyz:JIO.
ATR stop-distance colors: green <2%, orange 2-5%, red >5%; red mark means the position is out of the money; stop prices are highlighted red. Open-risk includes the funding reserve but excludes fees, slippage, and gaps; the stop trigger itself remains a price level.


Asset,Side,ATR (%),ATR Period Used,ATR Stop Distance (%),Funding Reserve / unit,Avg Entry,Mark,Stop,Take Profit,Open - Target (coins),Open - Target (%),Open Risk (price only) ($),Open Risk incl. funding ($)
ADA,SHORT,6.29%,14 days,7.86%,$0.000354086,0.193547,0.19233,0.20744,0.17722,"10,595.2",-90.70%,$16.41,$16.80
AERO,LONG,9.86%,14 days,12.33%,$0.000913248,0.832395,0.62746,0.5501,0.70482,"-2,189.19",-94.84%,$9.21,$9.31
AVAX,SHORT,6.34%,14 days,7.93%,$0.00300271,8.7597,8.6443,9.3299,7.9587,232.393,-88.57%,$20.57,$20.66
BTC,LONG,2.52%,14 days,3.15%,$113.752,"78,425","78,317","75,852","80,782",-0.0624362,-89.10%,$18.83,$19.70
FOGO,SHORT,14.41%,14 days,18.01%,$0,0.002597,0.002578,0.0030423,0.0021137,"309,688",-79.59%,$36.88,$36.88
GRASS,SHORT,12.81%,14 days,16.02%,$0,0.60132,0.60093,0.69718,0.50468,"1,424.75",-75.90%,$43.54,$43.54
IMX,SHORT,8.85%,14 days,11.07%,$0,0.12332,0.12135,0.13478,0.10792,"5,412.93",-40.23%,$107.98,$107.98
INIT,SHORT,15.53%,14 days,19.41%,$0.000790273,0.18359,0.175018,0.209,0.14104,"4,091.58",-78.74%,$37.55,$38.42
LIT,SHORT,11.32%,14 days,14.15%,$0.00364152,3.2512,3.214111,3.6688,2.7595,385.236,-97.72%,$4.09,$4.12
MELANIA,SHORT,7.50%,14 days,9.38%,$5.7346e-05,0.07686,0.07656,0.083741,0.069379,"19,284.8",-77.26%,$40.75,$41.08


Metric,Value
Wallets,2
Position value,"$8,383"
uPnL,$-106.87
ROE,-5.08%
Margin,"$2,104"
Margin % bal,56.54%
Available margin (80% of equity),"$2,890.79"
Available margin used (20% reserve),72.77%
Available margin remaining,$787.15
Available margin remaining (%),27.23%


ROE is unrealized PnL divided by used margin; a negative ROE means the open positions are currently at an unrealized loss relative to the margin supporting them.


## Realized PnL statistics

Pool spot/perp fills and signed funding across wallets.


In [8]:
analytics.show_realized_pnl()


=== REALIZED SPOT ===


Metric,Value
Realized PnL,$22.35 (0.62%)
24H,$0.00 (0.00%)
7D,$0.00 (0.00%)
30D,$22.35 (0.62%)
QTD,$22.35 (0.62%)
YTD,$22.35 (0.62%)
W,3
Win %,100.00%
Avg W,$7.45
L,0


=== REALIZED PERP ===


Metric,Value
Realized PnL,$-63.88 (-1.77%)
24H,$16.18 (0.45%)
7D,$354.61 (9.81%)
30D,$460.19 (12.74%)
QTD,$460.04 (12.73%)
YTD,$459.92 (12.73%)
W,189
Win %,68.23%
Avg W,$8.40
L,88



=== PERP PNL BREAKDOWN ===


Component,Amount ($),Contribution (% of |net PnL|)
Trading,$94.20,147.46%
Fees,$-106.95,-167.43%
Funding,$-51.13,-80.04%
Net perp PnL,$-63.88,


Liquidations: 0 of 277 closes (0.00%) | liquidation PnL: $0.00.


## Traded volume vs PnL

Combined turnover efficiency and correlations.


In [9]:
analytics.show_volume_summary()


=== TRADED VOLUME VS REALIZED PNL (ALL WALLETS) ===


Market,Volume ($),Realized PnL ($),Realized PnL / Volume (%),PnL / $1k volume ($),Active days,Daily Pearson r,Daily Spearman r
Perps,"$258,694.54",$-12.75,-0.00%,$-0.05,57,0.102,0.287
Spot,"$1,397.53",$22.35,1.60%,$16.00,5,-0.509,-0.462


## PnL attribution

Combined realized and open-position PnL by asset.


In [10]:
analytics.show_pnl_attribution()


=== CONSOLIDATED PNL ATTRIBUTION ===


Type,Asset,Closes,Realized,Realized (% gross),uPnL,uPnL (% gross),Total PnL,Total (% gross)
Perp,BTC,11,$-474.45,-18.87%,$-0.87,-0.35%,$-475.32,-17.49%
Perp,kBONK,3,$-397.06,-15.80%,$0.00,0.00%,$-397.06,-14.61%
Perp,ZORA,21,$304.49,12.11%,$0.00,0.00%,$304.49,11.20%
Perp,KAITO,4,$-193.84,-7.71%,$0.00,0.00%,$-193.84,-7.13%
Perp,HYPE,11,$146.54,5.83%,$0.00,0.00%,$146.54,5.39%
Perp,ETH,14,$116.62,4.64%,$0.00,0.00%,$116.62,4.29%
Perp,VVV,40,$116.26,4.62%,$0.00,0.00%,$116.26,4.28%
Perp,xyz:GOOGL,0,$-2.66,-0.11%,$-100.24,-39.90%,$-102.90,-3.79%
Perp,io:ANTH,2,$-86.69,-3.45%,$0.00,0.00%,$-86.69,-3.19%
Perp,AAVE,4,$54.17,2.16%,$0.00,0.00%,$54.17,1.99%


Net attributed PnL across wallets: $-148.38.
